In [ ]:
!pip install pyngrok

In [ ]:
from google.colab import output
import http.server, socketserver, threading, os, json
from urllib.request import urlopen, Request
from pyngrok import ngrok

BACKEND_URL = "url output from chatbotg2 file"

os.system("fuser -k 8501/tcp")
import time; time.sleep(1)

# Store server reference so it can be shut down
server_ref = {"httpd": None}

class ProxyHandler(http.server.SimpleHTTPRequestHandler):
    def do_POST(self):
        if self.path == "/proxy_query":
            length = int(self.headers["Content-Length"])
            body = self.rfile.read(length)
            try:
                req = Request(f"{BACKEND_URL}/run_query", data=body,
                              headers={"Content-Type": "application/json"})
                resp = urlopen(req)
                result = resp.read()
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Access-Control-Allow-Origin", "*")
                self.end_headers()
                self.wfile.write(result)
            except Exception as e:
                self.send_response(500)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps({"detail": str(e)}).encode())

        elif self.path == "/proxy_reset":
            try:
                req = Request(f"{BACKEND_URL}/reset", data=b"{}",
                              headers={"Content-Type": "application/json"})
                urlopen(req)
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Access-Control-Allow-Origin", "*")
                self.end_headers()
                self.wfile.write(b'{"status": "reset successful"}')
            except Exception as e:
                self.send_response(500)
                self.end_headers()

        elif self.path == "/shutdown":
            # Send response first, then shut down
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Access-Control-Allow-Origin", "*")
            self.end_headers()
            self.wfile.write(b'{"status": "shutting down"}')

            # Kill everything in a separate thread so response is sent first
            def kill():
                time.sleep(0.5)
                print("\n Shutdown triggered by user. Killing tunnels and server...")
                ngrok.kill()
                if server_ref["httpd"]:
                    server_ref["httpd"].shutdown()
                print(" All tunnels closed. You can close this tab.")
            threading.Thread(target=kill, daemon=True).start()

    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "POST, GET, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.end_headers()

    def log_message(self, format, *args):
        pass

os.chdir("/content")
import shutil
shutil.copy("/content/dashboard_assistant_ui.html", "/content/dashboard_ui_served.html")

def serve():
    with socketserver.TCPServer(("", 8501), ProxyHandler) as httpd:
        server_ref["httpd"] = httpd
        httpd.serve_forever()

threading.Thread(target=serve, daemon=True).start()
output.serve_kernel_port_as_iframe(8501, path="/dashboard_ui_served.html", height=500, width = "75%")

<IPython.core.display.Javascript object>